# Custom YOLOv8 Vehicle & Number Plate Detection

Welcome to the interactive model training walkthrough for **Number Plate & Vehicles Detection**. This Jupyter Notebook demonstrates transfer learning using Ultralytics YOLOv8 on custom classes:

1. **Bike**
2. **Bus**
3. **Car**
4. **Number plate**
5. **Person**
6. **Truck**

---

## Step 1: Install Dependencies

Let's install the Ultralytics framework and other necessary dependencies.

In [ ]:
# Install ultralytics, easyocr, and pytesseract wrappers
!pip install ultralytics easyocr pytesseract pandas matplotlib seaborn opencv-python pillow

## Step 2: Validate System Environment

Ensure PyTorch has GPU/CUDA capability for accelerated deep learning.

In [ ]:
import torch
import ultralytics

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
print(f"Ultralytics Version: {ultralytics.__version__}")

## Step 3: Split Dataset & Validate Labels

We will automatically split the flat source directory into standard YOLO splits `(train/val)`, validate annotations to clean anomalies, and filter out corrupted images.

In [ ]:
import os
import sys

# Add root project folder to python system path to resolve local imports
sys.path.append(os.path.abspath('..'))

from utils.synthetic_data import split_and_prepare_dataset, write_data_yaml

src_dir = r"C:\Users\pssin\Desktop\opencv\DATASET-20260524T053828Z-3-001\DATASET"
dst_dir = "../dataset"

stats = split_and_prepare_dataset(src_dir, dst_dir)
yaml_path = write_data_yaml(dst_dir)

print(f"Dataset split completed: {stats}")
print(f"Created data configuration at: {yaml_path}")

## Step 4: Exploratory Data Analysis (EDA)

Let's look at the class distribution and properties of our custom dataset.

In [ ]:
from utils.eda_helpers import get_annotations_dataframe, plot_class_distribution, plot_bbox_size_distribution

# Generate labels dataframe
df_train = get_annotations_dataframe(dst_dir, split='train')
print(df_train.head())

# Plot class distribution
plot_class_distribution(df_train)

# Plot bounding box sizes
plot_bbox_size_distribution(df_train)

## Step 5: Start custom YOLOv8 Model Training

We use YOLOv8 Nano (`yolov8n.pt`) as our transfer learning base for extreme fast real-time frame rates. Let's trigger training via Python script or CLI command.

In [ ]:
# Run custom training for 30 epochs with batch size 8
!python ../train.py --data ../data.yaml --epochs 30 --batch 8 --model yolov8n.pt

## Step 6: Verify Training Results & Outputs

After training finishes, YOLO saves metric curves, precision charts, confusion matrices, and the model weights (`best.pt`, `last.pt`) under `runs/` and copies them to the `/weights` folder.

In [ ]:
import matplotlib.pyplot as plt
import cv2

results_png = '../runs/vehicle_plate_detect/results.png'
if os.path.exists(results_png):
    img = cv2.imread(results_png)
    plt.figure(figsize=(14, 10))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()
else:
    print("Results curves not found. Make sure training completes first!")

## Step 7: Test Inference on Sample Image

Let's load the newly trained model weights (`weights/best.pt`) and perform vehicle detection and license plate OCR!

In [ ]:
from detect import load_yolo_model, run_inference_on_image
import matplotlib.pyplot as plt
import cv2

# Load weights
model = load_yolo_model('../weights/best.pt')

# Locate an image from validation split
val_images_dir = '../dataset/images/val'
sample_files = os.listdir(val_images_dir)

if sample_files:
    sample_path = os.path.join(val_images_dir, sample_files[0])
    print(f"Running inference on: {sample_path}")
    
    annotated_img, detections, metrics = run_inference_on_image(sample_path, model, conf_threshold=0.3)
    
    # Display image
    plt.figure(figsize=(12, 9))
    plt.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()
    
    # Print metrics
    print("Detections Metadata:", detections)
    print("Performance metrics:", metrics)
else:
    print("Validation set is empty. Perform Step 3 first!")